# M1.6: Reproducible energy modelling projects

## Objectives

At the end of this notebook you should be able to:

- lay out a modelling project so that someone else can run it from a fresh clone: package, config, data, tests, results
- keep every model number in a **config file** with units, and validate it before the model runs
- write **tests that check physics and economics**, and know why a test must compare against an independent source
- run a model from the **command line**, record the environment, and keep secrets and large data out of Git
- write a README that a hiring manager or reviewer will actually read

This lesson starts **portfolio project P1** (see `../PROJECT_P1.md`). You know Git, packaging and testing from software work — what is specific here is *what* to test in an energy model, how to make results reproducible, and how a modelling repo should read to a reviewer. Everything below runs on the real template in `template/`, which you will copy as the starting point of P1.

In [1]:
import os
import shutil
import subprocess
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import yaml

TEMPLATE = Path("template").resolve()
assert (TEMPLATE / "pyproject.toml").is_file(), "run this notebook from 01_python_for_energy_modelling/11_project_structure/"

def run(cmd: list[str], cwd: Path, env: dict | None = None) -> str:
    """Run a command in `cwd` and return its combined output."""
    out = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, env=env)
    return out.stdout + out.stderr

### Anatomy of a modelling project

A notebook is a great place to *think*; it is a poor place to keep the model. The template splits responsibilities:

In [2]:
skip = {"__pycache__", ".pytest_cache"}
for path in sorted(TEMPLATE.rglob("*")):
    if path.is_file() and not skip & set(path.parts) and path.suffix != ".pyc":
        print(path.relative_to(TEMPLATE))

.gitignore
README.md
config/config.yaml
data/README.md
data/processed/.gitkeep
data/raw/.gitkeep
environment.yml
notebooks/README.md
pyproject.toml
results/.gitkeep
src/hr_energy/__init__.py
src/hr_energy/__main__.py
src/hr_energy/cli.py
src/hr_energy/config.py
src/hr_energy/data.py
src/hr_energy/model.py
tests/conftest.py
tests/test_cli.py
tests/test_config.py
tests/test_data.py
tests/test_model.py


| Folder | Role | Rule |
|---|---|---|
| `config/` | every number the model uses, with units | change scenarios here, never in code |
| `data/` | `raw/` downloads, `processed/` inputs, `README.md` provenance | data are re-created by code, so they stay out of Git; the README says where they came from |
| `src/hr_energy/` | the code, as an installable package | notebooks import from it instead of copying functions |
| `tests/` | automated checks of physics and economics | must pass before any result is trusted |
| `notebooks/` | exploration and figures | thin: call the package, show results |
| `results/` | generated outputs | git-ignored, reproduced by one command |

This is the **src layout**: code lives in `src/hr_energy/` and is installed with `pip install -e .` (the `-e` makes edits take effect immediately). It prevents a classic bug: tests passing on your machine because they import a stray file from the working directory instead of the package.

### Every number in a config file

In M1.2 the capacities and costs lived inside `build_network()`. That works for one run, but a scenario study needs many variants, and a reviewer needs to see every assumption in one place. The template keeps them in YAML, with units in the comments:

In [3]:
print((TEMPLATE / "config" / "config.yaml").read_text())

# Every number the model uses lives here, with its unit. Change scenarios here, not in code.
name: base

snapshots:
  start: "2025-01-13"     # UTC, interval start
  hours: 168

load:
  source: synthetic       # "synthetic" (offline) or a CSV path under data/processed/
  mean_MW: 2100
  daily_amplitude_MW: 380
  weekend_drop_MW: 200
  seed: 1

generators:               # p_nom in MW, marginal_cost in EUR/MWh, p_max_pu as constant availability
  nuclear_share: {carrier: nuclear, p_nom: 350,  marginal_cost: 10}
  hydro_ror:     {carrier: hydro,   p_nom: 1500, marginal_cost: 2,   p_max_pu: 0.45}
  coal:          {carrier: coal,    p_nom: 210,  marginal_cost: 95}
  gas_ccgt:      {carrier: gas,     p_nom: 700,  marginal_cost: 110}
  gas_ocgt:      {carrier: gas,     p_nom: 300,  marginal_cost: 160}
  imports:       {carrier: imports, p_nom: 1500, marginal_cost: 120}

solver: highs



The config is **validated before the model runs**. A negative capacity or an availability above 1 would otherwise produce a model that solves happily and returns nonsense. Failing early, with a message that names the problem, saves hours:

In [4]:
sys.path.insert(0, str(TEMPLATE / "src"))
from hr_energy.config import ConfigError, load_config, validate_config

config = load_config(TEMPLATE / "config" / "config.yaml")
config["generators"]["hydro_ror"]["p_max_pu"] = 4.5        # someone typed 4.5 instead of 0.45
try:
    validate_config(config)
except ConfigError as err:
    print(f"ConfigError: {err}")

ConfigError: generator 'hydro_ror': p_max_pu must be in [0, 1]


### Tests that check physics and economics

Unit tests in software check that code does what it says. Tests in a **model** must also check that the results obey physics and economics — the validation habit from every lesson so far, made automatic. The template's tests check:

- supply equals demand in every hour (Kirchhoff's current law at the bus)
- the objective equals Σ p × marginal cost, recomputed by hand
- every price equals the merit-order price (single bus, no storage)
- the built network matches the config, and no plant exceeds its available capacity
- an impossible system **fails loudly** instead of returning numbers
- energy by carrier adds up to total load (units: MW × h = MWh)

Run them exactly as you would on the command line:

In [5]:
print(run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=TEMPLATE)[-300:])

..............                                                           [100%]
14 passed in 5.51s



All tests pass. `pytest` finds the package without installing it, because `pyproject.toml` sets `pythonpath = ["src"]` — a fresh clone is testable immediately.

### A test must compare against an independent source

Do the tests actually catch mistakes? Find out the way professionals do: **break the code on purpose** and check that a test fails. Copy the template, make the model ignore hydro's 45 % availability, and run the tests:

In [6]:
with tempfile.TemporaryDirectory() as tmp:
    broken = Path(tmp) / "broken"
    shutil.copytree(TEMPLATE, broken, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    model_py = broken / "src" / "hr_energy" / "model.py"
    model_py.write_text(model_py.read_text().replace('p_max_pu=g.get("p_max_pu", 1.0))', "p_max_pu=1.0)"))
    report = run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=broken)
print("\n".join(line for line in report.splitlines() if "FAILED" in line or "passed" in line))

FAILED tests/test_model.py::test_network_matches_config - AssertionError: ass...
1 failed, 13 passed in 5.50s


Exactly one test fails: `test_network_matches_config`. Notice which tests did **not** fail. The merit-order test reads availability from the network the code built — so when the code gets availability wrong, the test inherits the same mistake and agrees with it. Only a test that compares against an **independent source** (here the config file) catches the bug.

When this template was written, that test did not exist yet, and this exact experiment showed all tests passing on broken code. Breaking your own code on purpose (*mutation testing*) is the fastest way to find out whether your tests protect anything.

### One command to reproduce the results

A reviewer should not need to open a notebook to reproduce your numbers. The package has a command-line entry point that reads a config, solves, validates and writes results:

In [7]:
# In your own clone you run `pip install -e .` once; here PYTHONPATH=src stands in for that install.
env = {**os.environ, "PYTHONPATH": str(TEMPLATE / "src")}
with tempfile.TemporaryDirectory() as tmp:
    print(run([sys.executable, "-m", "hr_energy", "--config", "config/config.yaml", "--out", tmp],
              cwd=TEMPLATE, env=env))
    summary = (Path(tmp) / "base" / "summary.json").read_text()
print(summary)

base: total cost 19,347,678 EUR, mean price 115.9 EUR/MWh -> /tmp/tmp8l635eau/base

{
  "total_cost_EUR": 19347678.0,
  "load_MWh": 342719.3,
  "mean_price_EUR_per_MWh": 115.89285714285714,
  "max_balance_gap_MW": 0.0,
  "cost_check_ok": true,
  "energy_by_carrier_MWh": {
    "coal": 35280.0,
    "gas": 104743.8,
    "hydro": 113400.0,
    "imports": 30495.5,
    "nuclear": 58800.0
  }
}


The summary stores the validation results next to the numbers (`max_balance_gap_MW`, `cost_check_ok`), so every result file proves it was checked. A different scenario is a different config file, e.g. `config/high_imports.yaml` with its own `name`, and its results land in `results/<name>/`.

### Environment, secrets and data

Results depend on library versions — PyPSA, pandas and HiGHS all change behaviour between releases (you have already met a PyPSA default that changes in version 2.0). Record what you ran with:

In [8]:
for pkg in ["pypsa", "linopy", "highspy", "pandas", "numpy", "pyyaml"]:
    print(f"{pkg:8s} {version(pkg)}")

pypsa    1.4.0
linopy   0.10.1
highspy  1.15.1
pandas   3.0.6
numpy    2.5.4
pyyaml   6.0.3


Three rules keep a repository safe and small:

1. **Pin** minimum versions in `pyproject.toml` / `environment.yml`, and write the exact versions you used in the README.
2. **Secrets** (your ENTSO-E token) live in `.env`, which `.gitignore` excludes. A token pushed to GitHub is leaked even if you delete it later — it stays in the history.
3. **Data** that code can re-download stay out of Git; `data/README.md` records source, retrieval date, licence and processing for each file:

In [9]:
print((TEMPLATE / ".gitignore").read_text())

# secrets
.env

# data you can re-download (keep data/README.md and the .gitkeep files)
data/raw/*
data/processed/*
!data/raw/.gitkeep
!data/processed/.gitkeep

# generated outputs
results/*
!results/.gitkeep

# Python
__pycache__/
*.egg-info/
.pytest_cache/
.ipynb_checkpoints/



### A README that gets read

Reviewers — hiring managers, supervisors — spend a minute on your README. The template's README is ordered for that minute: **question and result first** (with the one figure that shows it), then method, validation, limitations, and how to reproduce. Limitations are not a weakness to hide: a model with stated limits is one a reviewer can trust.

In [10]:
readme = (TEMPLATE / "README.md").read_text()
print([line for line in readme.splitlines() if line.startswith("## ")])

['## Question and result', '## Method', '## Validation', '## Limitations', '## Reproduce', '## Project layout', '## Data and licence']


Your P1 repository starts as a copy of this template. The brief in `../PROJECT_P1.md` says what to build into it.

## Check your understanding

**Structure**

1. Why should a notebook import `build_network` from the package instead of defining its own copy?
2. A colleague wants to test a scenario with 500 MW more solar. Which files change, and which must not?

**Testing**

1. Write down one more bug that the current tests would **not** catch. Then sketch the test that would — and say what independent source it compares against.
2. Why does `test_infeasible_system_fails_loudly` matter more for a model than for most software?

**Reproducibility**

1. Six months from now, PyPSA 2.0 changes a default and your costs shift by 0.1 %. What in this project lets you find out why?
2. Your `data/raw/` folder holds a 300 MB ENTSO-E download. How does someone else get the same file?